# Module 4: Attention 机制

## 学习目标
- 理解标准 Attention 的计算过程
- 了解 FlashAttention 的优化原理
- 掌握 Paged Attention 的概念
- 学习 Mini-SGLang 的 Attention Backend 设计

---

## 4.1 代码位置

Attention 相关代码位于 `python/minisgl/attention/` 目录：

```
mini-sglang/python/minisgl/attention/
├── base.py   # 抽象基类定义
├── fa3.py    # FlashAttention3 后端
├── fi.py     # FlashInfer 后端
└── utils.py  # 工具函数
```

## 4.2 标准 Self-Attention

### 公式:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

其中:
- $Q$: Query 矩阵 $[seq\_len, d_k]$
- $K$: Key 矩阵 $[seq\_len, d_k]$  
- $V$: Value 矩阵 $[seq\_len, d_v]$
- $d_k$: Key/Query 的维度

In [ ]:
import torch
import torch.nn.functional as F
import math

def standard_attention(
    q: torch.Tensor,  # [batch, num_heads, seq_len, head_dim]
    k: torch.Tensor,  # [batch, num_heads, seq_len, head_dim]
    v: torch.Tensor,  # [batch, num_heads, seq_len, head_dim]
    causal: bool = True,
) -> torch.Tensor:
    """标准的 Scaled Dot-Product Attention"""
    batch_size, num_heads, seq_len, head_dim = q.shape
    scale = 1.0 / math.sqrt(head_dim)
    
    # 计算 attention scores: Q @ K^T / sqrt(d_k)
    # [batch, num_heads, seq_len, seq_len]
    attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scale
    
    # 因果遮罩 (causal mask): 防止看到未来的 token
    if causal:
        mask = torch.triu(torch.ones(seq_len, seq_len, device=q.device), diagonal=1)
        attn_weights = attn_weights.masked_fill(mask.bool(), float('-inf'))
    
    # Softmax 归一化
    attn_weights = F.softmax(attn_weights, dim=-1)
    
    # 加权求和: attention_weights @ V
    output = torch.matmul(attn_weights, v)
    
    return output, attn_weights

# 测试
batch_size = 2
num_heads = 8
seq_len = 16
head_dim = 64

q = torch.randn(batch_size, num_heads, seq_len, head_dim)
k = torch.randn(batch_size, num_heads, seq_len, head_dim)
v = torch.randn(batch_size, num_heads, seq_len, head_dim)

output, attn_weights = standard_attention(q, k, v, causal=True)

print(f"输入形状: Q={q.shape}, K={k.shape}, V={v.shape}")
print(f"Attention weights 形状: {attn_weights.shape}")
print(f"输出形状: {output.shape}")

In [ ]:
# 可视化因果遮罩
import matplotlib.pyplot as plt

# 取一个样本的 attention weights
sample_weights = attn_weights[0, 0].detach().numpy()  # [seq_len, seq_len]

plt.figure(figsize=(10, 8))
plt.imshow(sample_weights, cmap='Blues')
plt.colorbar(label='Attention Weight')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Causal Attention Pattern\n(下三角: 每个位置只能看到自己和之前的位置)')
plt.show()

print("观察: 因果遮罩使得每个 token 只能 attend 到自己和之前的 token")

## 4.3 标准 Attention 的问题

### 内存复杂度: $O(n^2)$

对于序列长度 $n$:
- Attention weights 矩阵需要 $n \times n$ 的内存
- 对于 $n = 128K$ tokens, 即使是 FP16 也需要 32GB!

```
内存 = n² × 2 bytes (FP16) × num_heads × batch_size
     = 128K² × 2 × 32 × 1
     = 1TB!
```

### 计算效率问题

标准实现需要:
1. 计算 $QK^T$ → 写入 HBM
2. 从 HBM 读取 → Softmax → 写入 HBM
3. 从 HBM 读取 → 乘以 V → 写入 HBM

**HBM (High Bandwidth Memory) 访问是瓶颈！**

In [ ]:
def attention_memory_usage(
    seq_len: int,
    num_heads: int,
    batch_size: int = 1,
    dtype_bytes: int = 2,  # FP16
) -> float:
    """计算 attention weights 的内存占用 (GB)"""
    size_bytes = seq_len * seq_len * num_heads * batch_size * dtype_bytes
    return size_bytes / (1024 ** 3)

# 不同序列长度的内存占用
seq_lens = [1024, 4096, 16384, 65536, 131072]
num_heads = 32

print("Attention Weights 内存占用 (FP16):")
print("="*50)
for seq_len in seq_lens:
    mem = attention_memory_usage(seq_len, num_heads)
    print(f"  seq_len={seq_len:>6}: {mem:>10.2f} GB")

print("\n结论: 长序列的 attention weights 需要巨大内存")

## 4.4 FlashAttention 原理

FlashAttention 通过 **分块计算 (Tiling)** 和 **重计算 (Recomputation)** 来解决内存问题。

### 核心思想:
1. **分块处理**: 将 Q, K, V 分成小块，每次只处理一小部分
2. **在 SRAM 中计算**: 利用 GPU 的高速 SRAM 进行计算
3. **在线 Softmax**: 边计算边更新 softmax 结果

```
标准 Attention:
┌─────────────────────────────────────┐
│  Q @ K^T (写入 HBM)                 │  ← HBM 写入
│       ↓                             │
│  从 HBM 读取                        │  ← HBM 读取
│       ↓                             │
│  Softmax (写入 HBM)                 │  ← HBM 写入
│       ↓                             │
│  从 HBM 读取                        │  ← HBM 读取
│       ↓                             │
│  @ V (写入 HBM)                     │  ← HBM 写入
└─────────────────────────────────────┘

FlashAttention:
┌─────────────────────────────────────┐
│  For each Q block:                  │
│    For each K, V block:             │
│      ┌───────────────────────────┐  │
│      │ 在 SRAM 中计算:           │  │
│      │   Q_block @ K_block^T     │  │
│      │   在线更新 Softmax        │  │
│      │   累加 @ V_block          │  │
│      └───────────────────────────┘  │
│  写入最终结果到 HBM                 │  ← 只有最终结果写入 HBM
└─────────────────────────────────────┘
```

In [ ]:
def online_softmax_update(
    m_prev: torch.Tensor,   # 之前的 max
    l_prev: torch.Tensor,   # 之前的 sum(exp)
    o_prev: torch.Tensor,   # 之前的输出
    m_new: torch.Tensor,    # 新块的 max
    l_new: torch.Tensor,    # 新块的 sum(exp)
    o_new: torch.Tensor,    # 新块的输出
):
    """
    在线 Softmax 更新算法
    
    用于合并两个块的 softmax 结果，无需存储完整的 attention matrix
    """
    # 更新 max
    m_updated = torch.maximum(m_prev, m_new)
    
    # 重新缩放因子
    scale_prev = torch.exp(m_prev - m_updated)
    scale_new = torch.exp(m_new - m_updated)
    
    # 更新 sum(exp)
    l_updated = scale_prev * l_prev + scale_new * l_new
    
    # 更新输出 (加权平均)
    o_updated = (scale_prev * l_prev * o_prev + scale_new * l_new * o_new) / l_updated
    
    return m_updated, l_updated, o_updated

# 模拟分块 attention 计算
def flash_attention_simulation(
    q: torch.Tensor,  # [seq_len, head_dim]
    k: torch.Tensor,  # [seq_len, head_dim]
    v: torch.Tensor,  # [seq_len, head_dim]
    block_size: int = 4,
):
    """FlashAttention 的简化模拟"""
    seq_len, head_dim = q.shape
    scale = 1.0 / math.sqrt(head_dim)
    
    # 输出和统计量
    output = torch.zeros_like(q)
    m = torch.full((seq_len, 1), float('-inf'))  # max values
    l = torch.zeros(seq_len, 1)  # sum of exp
    
    num_blocks = (seq_len + block_size - 1) // block_size
    
    for i in range(num_blocks):  # 遍历 Q 的块
        q_start = i * block_size
        q_end = min(q_start + block_size, seq_len)
        q_block = q[q_start:q_end]
        
        for j in range(i + 1):  # 只处理因果范围内的 K, V 块
            k_start = j * block_size
            k_end = min(k_start + block_size, seq_len)
            k_block = k[k_start:k_end]
            v_block = v[k_start:k_end]
            
            # 计算块内的 attention scores
            scores = torch.matmul(q_block, k_block.t()) * scale
            
            # 因果遮罩 (如果是对角块)
            if i == j:
                mask = torch.triu(
                    torch.ones(q_end - q_start, k_end - k_start),
                    diagonal=1
                )
                scores = scores.masked_fill(mask.bool(), float('-inf'))
            
            # 计算块的 max 和 exp
            m_block = scores.max(dim=-1, keepdim=True)[0]
            exp_scores = torch.exp(scores - m_block)
            l_block = exp_scores.sum(dim=-1, keepdim=True)
            o_block = torch.matmul(exp_scores, v_block)
            
            # 在线更新
            m_old = m[q_start:q_end]
            l_old = l[q_start:q_end]
            o_old = output[q_start:q_end]
            
            m_new, l_new, o_new = online_softmax_update(
                m_old, l_old, o_old,
                m_block, l_block, o_block
            )
            
            m[q_start:q_end] = m_new
            l[q_start:q_end] = l_new
            output[q_start:q_end] = o_new
    
    return output

# 测试
seq_len = 16
head_dim = 8
q = torch.randn(seq_len, head_dim)
k = torch.randn(seq_len, head_dim)
v = torch.randn(seq_len, head_dim)

# FlashAttention 模拟
output_flash = flash_attention_simulation(q, k, v, block_size=4)

# 标准 Attention 对比
q_4d = q.unsqueeze(0).unsqueeze(0)
k_4d = k.unsqueeze(0).unsqueeze(0)
v_4d = v.unsqueeze(0).unsqueeze(0)
output_std, _ = standard_attention(q_4d, k_4d, v_4d, causal=True)
output_std = output_std.squeeze()

print(f"FlashAttention 输出形状: {output_flash.shape}")
print(f"标准 Attention 输出形状: {output_std.shape}")
print(f"\n两者差异 (应该非常小): {(output_flash - output_std).abs().max():.6f}")

## 4.5 Paged Attention

Paged Attention 借鉴了操作系统的虚拟内存概念，用于高效管理 KV Cache。

### 核心思想:
- 将 KV Cache 分成固定大小的 "页" (pages)
- 使用 "页表" (page table) 来映射逻辑位置到物理位置
- 支持非连续存储，提高内存利用率

```
传统 KV Cache (连续存储):
┌──────────────────────────────────────────────────────────────┐
│ Req 0: [K0, K1, K2, K3, K4, ...]  ← 需要预分配最大长度      │
│ Req 1: [K0, K1, K2, ...]          ← 浪费很多空间            │
└──────────────────────────────────────────────────────────────┘

Paged Attention (分页存储):
┌──────────────────────────────────────────────────────────────┐
│ 物理页池:                                                    │
│   Page 0: [K]  Page 1: [K]  Page 2: [K]  Page 3: [K] ...   │
│                                                              │
│ 页表:                                                        │
│   Req 0: [0, 2, 5, 7, ...]  ← 动态分配，按需增长            │
│   Req 1: [1, 3, 4, ...]     ← 高效利用内存                  │
└──────────────────────────────────────────────────────────────┘
```

In [ ]:
class PagedKVCache:
    """简化版的 Paged KV Cache 实现"""
    
    def __init__(
        self,
        num_pages: int,
        page_size: int,
        num_layers: int,
        num_heads: int,
        head_dim: int,
        dtype: torch.dtype = torch.float16,
    ):
        self.num_pages = num_pages
        self.page_size = page_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.head_dim = head_dim
        
        # 分配物理页池
        # [num_layers, num_pages, page_size, num_heads, head_dim]
        self.k_cache = torch.zeros(
            num_layers, num_pages, page_size, num_heads, head_dim, dtype=dtype
        )
        self.v_cache = torch.zeros(
            num_layers, num_pages, page_size, num_heads, head_dim, dtype=dtype
        )
        
        # 空闲页列表
        self.free_pages = list(range(num_pages))
        
        print(f"PagedKVCache 初始化:")
        print(f"  总页数: {num_pages}")
        print(f"  每页大小: {page_size} tokens")
        print(f"  总容量: {num_pages * page_size} tokens")
    
    def allocate_pages(self, num_pages: int) -> list:
        """分配指定数量的页"""
        if num_pages > len(self.free_pages):
            raise RuntimeError(f"Not enough free pages: need {num_pages}, have {len(self.free_pages)}")
        
        allocated = [self.free_pages.pop(0) for _ in range(num_pages)]
        return allocated
    
    def free_pages(self, pages: list) -> None:
        """释放页"""
        self.free_pages.extend(pages)
    
    def store_kv(
        self,
        layer_id: int,
        page_indices: torch.Tensor,  # 物理页索引
        k: torch.Tensor,  # [num_tokens, num_heads, head_dim]
        v: torch.Tensor,  # [num_tokens, num_heads, head_dim]
    ):
        """存储 K, V 到指定的页"""
        num_tokens = k.shape[0]
        for i in range(num_tokens):
            page_idx = page_indices[i].item()
            slot_idx = i % self.page_size  # 页内偏移
            self.k_cache[layer_id, page_idx, slot_idx] = k[i]
            self.v_cache[layer_id, page_idx, slot_idx] = v[i]

# 创建示例
cache = PagedKVCache(
    num_pages=64,
    page_size=1,  # Mini-SGLang 使用 page_size=1
    num_layers=12,
    num_heads=8,
    head_dim=64,
)

## 4.6 Mini-SGLang 的 Attention Backend

Mini-SGLang 定义了一个抽象的 Attention Backend 接口：

```python
class BaseAttnBackend(ABC):
    @abstractmethod
    def forward(self, q, k, v, layer_id, batch) -> torch.Tensor: ...
    
    @abstractmethod
    def prepare_metadata(self, batch) -> None: ...
    
    @abstractmethod
    def init_capture_graph(self, max_seq_len, bs_list) -> None: ...
    
    @abstractmethod
    def prepare_for_capture(self, batch) -> None: ...
    
    @abstractmethod
    def prepare_for_replay(self, batch) -> None: ...
```

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List

@dataclass
class BaseAttnMetadata(ABC):
    """Attention 元数据基类"""
    positions: torch.Tensor  # 位置索引
    
    @abstractmethod
    def get_last_indices(self, bs: int) -> torch.Tensor:
        """获取每个序列最后一个 token 的索引"""
        ...

class BaseAttnBackend(ABC):
    """Attention 后端基类"""
    
    @abstractmethod
    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        layer_id: int,
        batch,  # Batch 对象
    ) -> torch.Tensor:
        """执行 attention 计算"""
        ...
    
    @abstractmethod
    def prepare_metadata(self, batch) -> None:
        """准备 attention 元数据"""
        ...
    
    @abstractmethod
    def init_capture_graph(self, max_seq_len: int, bs_list: List[int]) -> None:
        """初始化 CUDA Graph 捕获"""
        ...

print("BaseAttnBackend 定义了 5 个抽象方法:")
print("  1. forward(): 执行 attention 计算")
print("  2. prepare_metadata(): 准备元数据")
print("  3. init_capture_graph(): 初始化 CUDA Graph")
print("  4. prepare_for_capture(): 准备 Graph 捕获")
print("  5. prepare_for_replay(): 准备 Graph 重放")

## 4.7 HybridBackend: Prefill 和 Decode 使用不同后端

Mini-SGLang 支持对 Prefill 和 Decode 阶段使用不同的 attention 后端：

- **Prefill**: 使用 FlashAttention3 (FA3)
  - 处理长序列，计算密集
  - FA3 针对这种场景优化

- **Decode**: 使用 FlashInfer (FI)
  - 处理单 token，内存密集
  - FI 支持 Paged KV Cache，针对解码优化

In [ ]:
class HybridBackend(BaseAttnBackend):
    """混合后端: Prefill 和 Decode 使用不同的后端"""
    
    def __init__(
        self,
        prefill_backend: BaseAttnBackend,
        decode_backend: BaseAttnBackend,
    ):
        self.prefill_backend = prefill_backend
        self.decode_backend = decode_backend
    
    def forward(self, q, k, v, layer_id, batch) -> torch.Tensor:
        # 根据 batch 的阶段选择后端
        backend = self.prefill_backend if batch.is_prefill else self.decode_backend
        return backend.forward(q, k, v, layer_id, batch)
    
    def prepare_metadata(self, batch) -> None:
        backend = self.prefill_backend if batch.is_prefill else self.decode_backend
        return backend.prepare_metadata(batch)
    
    def init_capture_graph(self, max_seq_len, bs_list) -> None:
        # CUDA Graph 只用于 decode
        self.decode_backend.init_capture_graph(max_seq_len, bs_list)

print("HybridBackend 使用策略:")
print("  Prefill: FlashAttention3 (处理长输入)")
print("  Decode: FlashInfer (处理单 token + Paged KV)")
print("\n这种设计可以针对不同阶段的特点进行优化")

## 4.8 FlashInfer 元数据

FlashInfer 需要一些特殊的元数据来进行 Paged Attention 计算：

In [ ]:
@dataclass
class FIMetadata:
    """FlashInfer 的元数据"""
    # 位置信息
    positions: torch.Tensor         # [total_tokens] 位置索引
    
    # 序列长度信息 (cumulative sum 形式)
    cu_seqlens_q_cpu: torch.Tensor  # [batch_size + 1] Query 的累积长度
    cu_seqlens_k_cpu: torch.Tensor  # [batch_size + 1] Key 的累积长度
    cu_seqlens_q_gpu: torch.Tensor  # GPU 上的副本
    
    # 页表信息
    indices: torch.Tensor           # [total_pages] 物理页索引
    last_page_len_cpu: torch.Tensor # [batch_size] 最后一页的有效长度
    
    # 模型配置
    num_qo_heads: int
    num_kv_heads: int
    head_dim: int
    page_size: int  # 目前只支持 1

# 模拟创建元数据
def create_fi_metadata_example():
    """创建示例 FI 元数据"""
    # 假设有 3 个请求，序列长度分别为 [5, 3, 7]
    seq_lens = [5, 3, 7]
    batch_size = len(seq_lens)
    
    # 计算累积长度
    cu_seqlens = torch.tensor([0] + seq_lens).cumsum(dim=0)
    print(f"序列长度: {seq_lens}")
    print(f"累积长度 (cu_seqlens): {cu_seqlens.tolist()}")
    
    # 生成位置索引
    positions = []
    for seq_len in seq_lens:
        positions.extend(range(seq_len))
    positions = torch.tensor(positions)
    print(f"位置索引: {positions.tolist()}")
    
    # 模拟页表索引 (假设每个 token 对应一个页)
    total_tokens = sum(seq_lens)
    page_indices = torch.arange(total_tokens)  # 简化: 每个 token 一个页
    print(f"页表索引: {page_indices.tolist()}")
    
    return {
        'positions': positions,
        'cu_seqlens': cu_seqlens,
        'page_indices': page_indices,
        'seq_lens': seq_lens,
    }

metadata = create_fi_metadata_example()

## 4.9 AttentionLayer 的完整流程

让我们看看 `AttentionLayer.forward()` 的完整流程：

In [ ]:
class AttentionLayer:
    """完整的 Attention 层实现 (简化版)"""
    
    def __init__(
        self,
        layer_id: int,
        num_qo_heads: int,
        num_kv_heads: int,
        head_dim: int,
        rotary_embedding,  # RoPE
    ):
        self.layer_id = layer_id
        self.num_qo_heads = num_qo_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        self.qo_attn_dim = num_qo_heads * head_dim
        self.kv_attn_dim = num_kv_heads * head_dim
        self.rotary = rotary_embedding
    
    def forward(self, qkv: torch.Tensor, ctx) -> torch.Tensor:
        """
        qkv: [total_tokens, qo_dim + 2 * kv_dim] 来自 QKV Linear
        ctx: 全局上下文，包含 batch 和 attn_backend
        """
        metadata = ctx.batch.attn_metadata
        
        # Step 1: 分割 Q, K, V
        q, k, v = qkv.split([self.qo_attn_dim, self.kv_attn_dim, self.kv_attn_dim], dim=-1)
        print(f"Step 1 - Split QKV:")
        print(f"  Q: {q.shape}, K: {k.shape}, V: {v.shape}")
        
        # Step 2: 应用 RoPE
        if self.rotary:
            q, k = self.rotary.forward(metadata.positions, q, k)
            print(f"Step 2 - Apply RoPE")
        
        # Step 3: Reshape for attention
        q = q.view(-1, self.num_qo_heads, self.head_dim)
        print(f"Step 3 - Reshape Q: {q.shape}")
        
        # Step 4: 调用 attention backend
        # backend 会:
        #   1. 将 K, V 存入 KV Cache
        #   2. 从 KV Cache 读取所有 K, V
        #   3. 计算 attention
        print(f"Step 4 - Call attn_backend.forward()")
        print(f"  - Store K, V to KV Cache")
        print(f"  - Read from KV Cache (current + cached)")
        print(f"  - Compute attention")
        
        # 模拟输出
        o = torch.randn_like(q)
        
        # Step 5: Reshape output
        return o.view(-1, self.qo_attn_dim)

# 模拟执行
print("=== AttentionLayer Forward Pass ===")
print()

# 创建模拟层
class MockMetadata:
    positions = torch.arange(10)

class MockBatch:
    attn_metadata = MockMetadata()

class MockContext:
    batch = MockBatch()

attn_layer = AttentionLayer(
    layer_id=0,
    num_qo_heads=8,
    num_kv_heads=2,  # GQA 4x
    head_dim=64,
    rotary_embedding=None,
)

# 模拟 QKV 输入
qkv_dim = 8 * 64 + 2 * 2 * 64  # Q + K + V
qkv = torch.randn(10, qkv_dim)  # 10 tokens

output = attn_layer.forward(qkv, MockContext())
print(f"\nFinal output: {output.shape}")

## 4.10 小结

### 核心要点:

1. **标准 Attention 的问题**:
   - 内存复杂度 $O(n^2)$
   - HBM 访问是瓶颈

2. **FlashAttention 优化**:
   - 分块计算 (Tiling)
   - 在线 Softmax 更新
   - 减少 HBM 访问

3. **Paged Attention**:
   - 将 KV Cache 分页存储
   - 使用页表进行地址映射
   - 提高内存利用率

4. **HybridBackend**:
   - Prefill: 使用 FlashAttention3
   - Decode: 使用 FlashInfer
   - 针对不同阶段优化

5. **AttentionLayer 流程**:
   - Split Q, K, V
   - Apply RoPE
   - Reshape
   - Call backend (store KV + compute attention)
   - Output

---

**下一步**: [Module 5: KV Cache 管理](./05_kv_cache_management.ipynb) - 深入学习 Naive Cache 和 Radix Cache。